# RQ2 — Evaluation: Robustness to Structural Perturbations

This notebook takes the `rq2_pairs.csv` (or `rq2_pairs_validated.csv`) produced by
**Notebook 1** and computes the metrics that fill **Table 4** of the paper.

### Models evaluated
| # | Model | Type |
|---|-------|------|
| 1 | TF-IDF | Bag-of-words baseline |
| 2 | SBERT/MiniLM (`all-MiniLM-L6-v2`) | Dense baseline |
| 3 | E5-base-v2 (`intfloat/e5-base-v2`) | Dense baseline |
| 4 | SBERT/MPNet (`all-mpnet-base-v2`) | Dense baseline |
| 5 | Qwen3-Emb-0.6B | LLM-based baseline |
| 6 | **SparqlCE** (yours) | Proposed model |

### Metric reported
- **Mean cosine similarity** per (model × perturbation) — primary metric  
- **Spearman ρ** vs. ground-truth Jaccard (= 1.0 for every pair) — supplementary  
  *(NaN when all ground-truth values are constant; mean cosine acts as proxy)*

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = Path("rq2_dataset")
OUTPUT_DIR = Path("rq2_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Use validated file if it exists, otherwise use the raw pairs
pairs_file = DATA_DIR / "rq2_pairs_validated.csv"
if not pairs_file.exists():
    pairs_file = DATA_DIR / "rq2_pairs.csv"
    print(f"⚠️  Validated file not found — using {pairs_file.name}")
    print("   Run endpoint validation in Notebook 1 for production results.")
else:
    print(f"✅ Using validated pairs: {pairs_file}")

# ── SparqlCE model path — UPDATE THIS ─────────────────────────────────────────
SPARQLCE_MODEL_PATH = "path/to/your/sparqlce_checkpoint"  # ← change to your checkpoint

PERTURBATION_ORDER = [
    "var_rename", "triple_reorder", "optional_insert", "limit_offset", "combined"
]

print("✅ Imports and config ready.")

In [ ]:
df = pd.read_csv(pairs_file)

# Keep only pending/valid pairs (drop skipped_no_change)
df = df[df["validation_status"] != "skipped_no_change"].copy()
df = df[df["validation_status"] != "DISCARD"].copy()

print(f"Loaded {len(df)} pairs across {df['dataset'].nunique()} dataset(s)")
print(df.groupby(["dataset", "perturbation"]).size().to_string())

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  Generic embedding interface — add new models by registering an encoder fn
# ────────────────────────────────────────────────────────────────────────────

ENCODERS = {}   # name -> callable(list[str]) -> np.ndarray  shape (n, dim)

# ── 1. TF-IDF ─────────────────────────────────────────────────────────────────
def make_tfidf_encoder(all_texts):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
    vec.fit(all_texts)
    def encode(texts):
        return vec.transform(texts).toarray().astype(np.float32)
    return encode

all_texts = pd.concat([df["original_query"], df["perturbed_query"]]).tolist()
ENCODERS["TF-IDF"] = make_tfidf_encoder(all_texts)
print("✅ TF-IDF encoder ready.")

In [ ]:
try:
    from sentence_transformers import SentenceTransformer

    ST_MODELS = {
        "SBERT/MiniLM" : "sentence-transformers/all-MiniLM-L6-v2",
        "E5-base-v2"   : "intfloat/e5-base-v2",
        "SBERT/MPNet"  : "sentence-transformers/all-mpnet-base-v2",
    }

    for name, model_id in ST_MODELS.items():
        print(f"  Loading {name} ({model_id})...")
        model = SentenceTransformer(model_id)
        # closure to capture model
        def make_encoder(m):
            def encode(texts, batch_size=64):
                return m.encode(texts, batch_size=batch_size,
                                show_progress_bar=False,
                                normalize_embeddings=True)
            return encode
        ENCODERS[name] = make_encoder(model)
        print(f"  ✅ {name} ready.")

except ImportError:
    print("⚠️  sentence-transformers not installed. Run: pip install sentence-transformers")
    print("   Skipping SBERT/MiniLM, E5-base-v2, SBERT/MPNet.")

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    print("Loading Qwen3-Emb-0.6B (this may take a moment)...")
    qwen = SentenceTransformer("Qwen/Qwen3-Embedding", model_kwargs={"torch_dtype": "auto"})

    def encode_qwen(texts, batch_size=16):
        return qwen.encode(
            texts, batch_size=batch_size,
            show_progress_bar=False,
            normalize_embeddings=True,
            prompt_name="query",
        )
    ENCODERS["Qwen3-Emb-0.6B"] = encode_qwen
    print("✅ Qwen3-Emb-0.6B ready.")

except Exception as e:
    print(f"⚠️  Could not load Qwen3-Emb: {e}")
    print("   Skipping Qwen3-Emb-0.6B.")

In [ ]:
# ── Replace this block with your actual SparqlCE loading code ─────────────────
try:
    from sentence_transformers import SentenceTransformer   # or your custom loader
    sparqlce_model = SentenceTransformer(SPARQLCE_MODEL_PATH)

    def encode_sparqlce(texts, batch_size=64):
        return sparqlce_model.encode(
            texts, batch_size=batch_size,
            show_progress_bar=False,
            normalize_embeddings=True,
        )
    ENCODERS["SparqlCE"] = encode_sparqlce
    print("✅ SparqlCE ready.")

except Exception as e:
    print(f"⚠️  SparqlCE not loaded: {e}")
    print("   Update SPARQLCE_MODEL_PATH at the top of Cell 1.")

print(f"\nActive encoders: {list(ENCODERS.keys())}")

In [ ]:
def compute_cosine_similarities(df: pd.DataFrame, encoder_fn) -> np.ndarray:
    """
    Encode all original and perturbed queries, return per-pair cosine similarity.
    Returns array of shape (len(df),).
    """
    orig_embs  = encoder_fn(df["original_query"].tolist())
    pert_embs  = encoder_fn(df["perturbed_query"].tolist())
    # Row-wise cosine similarity
    sims = np.einsum("ij,ij->i",
                     orig_embs / (np.linalg.norm(orig_embs, axis=1, keepdims=True) + 1e-9),
                     pert_embs / (np.linalg.norm(pert_embs, axis=1, keepdims=True) + 1e-9))
    return sims.astype(np.float64)


results = {}   # model_name -> pd.Series with pair_id index

for model_name, encoder_fn in ENCODERS.items():
    print(f"Encoding with {model_name}...")
    sims = compute_cosine_similarities(df, encoder_fn)
    df[f"cos_{model_name}"] = sims
    print(f"  mean cos sim = {sims.mean():.4f}  |  std = {sims.std():.4f}")

print("\n✅ All cosine similarities computed.")

In [ ]:
def spearman_rho(sims: np.ndarray, gt: np.ndarray) -> float:
    """
    Spearman ρ between model cosine similarities and ground-truth Jaccards.
    Returns NaN when ground truth is constant (all 1.0).
    """
    if np.std(gt) < 1e-9:
        return float("nan")   # constant ground truth → undefined
    rho, _ = spearmanr(sims, gt)
    return rho


# ── Build Table 4 rows ────────────────────────────────────────────────────────
table4_rows = []

for model_name in ENCODERS.keys():
    col = f"cos_{model_name}"
    if col not in df.columns:
        continue

    row = {"Model": model_name}

    for pert in PERTURBATION_ORDER:
        sub = df[df["perturbation"] == pert]
        if len(sub) == 0:
            row[pert] = float("nan")
            continue
        sims = sub[col].values
        gt   = sub["ground_truth_jaccard"].values
        # Primary metric: mean cosine similarity
        mean_cos = sims.mean()
        rho      = spearman_rho(sims, gt)
        # Report mean cosine; annotate with ρ if defined
        row[pert]               = round(mean_cos, 4)
        row[f"{pert}_rho"]      = round(rho, 4) if not np.isnan(rho) else "—"

    # Average across all perturbations (mean cosine)
    row["Avg"] = round(
        df[col].values.mean(), 4
    )
    table4_rows.append(row)

df_table4 = pd.DataFrame(table4_rows)

# ── Pretty display (mean cosine only) ─────────────────────────────────────────
display_cols = ["Model"] + PERTURBATION_ORDER + ["Avg"]
display_cols = [c for c in display_cols if c in df_table4.columns]

col_rename = {
    "var_rename"     : "Var Rename",
    "triple_reorder" : "Triple Reorder",
    "optional_insert": "OPTIONAL Insert",
    "limit_offset"   : "LIMIT/OFFSET",
    "combined"       : "Combined",
    "Avg"            : "Avg",
}

table4_pretty = df_table4[display_cols].rename(columns=col_rename)
print("\n===  TABLE 4 — Mean Cosine Similarity (↑ better)  ===")
print(table4_pretty.to_string(index=False))

In [ ]:
# ── Supplementary: Spearman ρ table ──────────────────────────────────────────
rho_rows = []
for model_name in ENCODERS.keys():
    col = f"cos_{model_name}"
    if col not in df.columns:
        continue
    row = {"Model": model_name}
    for pert in PERTURBATION_ORDER:
        key = f"{pert}_rho"
        # Already computed above
        row[pert] = df_table4.loc[df_table4["Model"]==model_name, key].values[0] \
                    if key in df_table4.columns else "—"
    rho_rows.append(row)

df_rho = pd.DataFrame(rho_rows).rename(columns=col_rename)
print("\n===  Supplementary — Spearman ρ (NaN = constant ground truth)  ===")
print(df_rho.to_string(index=False))
print("\nNote: '—' means ground-truth is constant (all 1.0) → mean cosine is the primary metric.")

In [ ]:
# ── Save Table 4 (mean cosine) ────────────────────────────────────────────────
t4_path = OUTPUT_DIR / "table4_mean_cosine.csv"
table4_pretty.to_csv(t4_path, index=False)
print(f"Saved Table 4 → {t4_path}")

# ── Save Spearman ρ supplement ────────────────────────────────────────────────
rho_path = OUTPUT_DIR / "table4_spearman_rho.csv"
df_rho.to_csv(rho_path, index=False)
print(f"Saved Spearman ρ → {rho_path}")

# ── Save raw per-pair scores (for detailed analysis) ─────────────────────────
raw_path = OUTPUT_DIR / "rq2_per_pair_scores.csv"
cos_cols = [c for c in df.columns if c.startswith("cos_")]
df[["dataset","query_id","perturbation","ground_truth_jaccard"] + cos_cols].to_csv(
    raw_path, index=False)
print(f"Saved per-pair scores → {raw_path}")

print("\n✅ All results saved.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 130})

cos_cols = [c for c in df.columns if c.startswith("cos_")]
model_names = [c.replace("cos_", "") for c in cos_cols]

fig, axes = plt.subplots(1, len(PERTURBATION_ORDER),
                          figsize=(4.5 * len(PERTURBATION_ORDER), 4.5),
                          sharey=True)

colors = plt.cm.tab10.colors

for ax, pert in zip(axes, PERTURBATION_ORDER):
    sub = df[df["perturbation"] == pert]
    means = [sub[f"cos_{m}"].mean() for m in model_names]
    bars = ax.barh(model_names, means, color=colors[:len(model_names)], edgecolor="white", height=0.6)
    ax.set_xlim(0, 1.05)
    ax.axvline(1.0, color="red", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.set_title(col_rename.get(pert, pert), fontweight="bold")
    ax.set_xlabel("Mean Cosine Sim")
    for bar, val in zip(bars, means):
        ax.text(min(val + 0.01, 1.02), bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=8)

fig.suptitle("RQ2 — Mean Cosine Similarity per Perturbation Type", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plot_path = OUTPUT_DIR / "table4_barchart.png"
plt.savefig(plot_path, bbox_inches="tight")
plt.show()
print(f"✅ Plot saved → {plot_path}")

In [ ]:
# ── Generate a ready-to-paste LaTeX version of Table 4 ───────────────────────
latex_rows = []
pert_labels = ["Var Rename", "Triple Reorder", "OPTIONAL Insert", "LIMIT/OFFSET", "Combined", "Avg"]

header = " & ".join(["\\textbf{Model}"] + [f"\\textbf{{{l}}}" for l in pert_labels]) + " \\\\"
latex_rows.append(header)
latex_rows.append("\\midrule")

for _, row in table4_pretty.iterrows():
    model = row["Model"]
    vals  = [str(row.get(l, "—")) for l in pert_labels]
    # Bold the best value per column
    line = f"{'\\textbf{SparqlCE}' if model == 'SparqlCE' else model} & " + " & ".join(vals) + " \\\\"
    latex_rows.append(line)

latex_str = "\n".join([
    "\\begin{table}[h]",
    "\\centering",
    "\\caption{RQ2 — Mean Cosine Similarity under Semantics-Preserving Perturbations}",
    "\\label{tab:rq2}",
    "\\begin{tabular}{l" + "c" * len(pert_labels) + "}",
    "\\toprule",
] + latex_rows + [
    "\\bottomrule",
    "\\end{tabular}",
    "\\end{table}"
])

latex_path = OUTPUT_DIR / "table4.tex"
with open(latex_path, "w") as f:
    f.write(latex_str)

print(latex_str)
print(f"\n✅ LaTeX table saved → {latex_path}")